# Audit del modello fondamentale v4

## tl;dr

Il dataset trimestrale contiene **13.488 snapshot di 74 emittenti**: 10.207 snapshot 10-Q e 3.281 snapshot 10-K. Non risultano duplicati alla grana ticker-mese, filing futuri, unità incoerenti o valute miste. La promozione complessiva resta bloccata: 1 mese non supera la baseline, 3 mesi non è confermato dall'holdout e soltanto 1 anno supera tutti i gate. L'universo corrente non usa ancora un security master storico, quindi il survivorship bias resta un caveat materiale.

## Context & Methods

Audit riproducibile del candidato `fundamental_return_model_v4_quarterly_candidate.json` e del relativo CSV point-in-time. Le feature devono esistere alla data dello snapshot; le label devono essere successive. Il gate di produzione è letto direttamente da `ml.model_governance`.

### Key Assumptions

- `ticker + snapshot_date` è la grana attesa.
- `target_*` è un rendimento totale raw perché `target_kind=raw`.
- Le partizioni recenti possono avere label mancanti perché non ancora mature.
- Nessuna conclusione è causale: il modello misura associazioni fuori campione.

## Data

In [ ]:
from pathlib import Path
import json
import pandas as pd

working = Path.cwd()
backend = working if (working / 'models').exists() else working.parent
models = backend / 'models'
dataset_path = models / 'fundamental_v4_quarterly_dataset.csv'
artifact_path = models / 'fundamental_return_model_v4_quarterly_candidate.json'
assert dataset_path.exists() and artifact_path.exists(), 'Eseguire dalla cartella backend o backend/notebooks'

data = pd.read_csv(dataset_path, low_memory=False)
artifact = json.loads(artifact_path.read_text(encoding='utf-8'))
for column in ('snapshot_date', 'accepted_at', 'report_date'):
    data[column] = pd.to_datetime(data[column], utc=True).dt.tz_localize(None)

profile = pd.DataFrame({
    'metrica': ['righe', 'emittenti', 'snapshot 10-Q', 'snapshot 10-K', 'data iniziale', 'data finale'],
    'valore': [
        len(data), data['ticker'].nunique(),
        int((data['filing_frequency'] == 'quarterly').sum()),
        int((data['filing_frequency'] == 'annual').sum()),
        data['snapshot_date'].min().date(), data['snapshot_date'].max().date(),
    ],
})
print(profile.to_string(index=False))

## Results

### Integrità temporale, grana e unità

In [ ]:
checks = {
    'duplicati ticker-mese': int(data.duplicated(['ticker', 'snapshot_date']).sum()),
    'filing accettati dopo lo snapshot': int((data['accepted_at'] > data['snapshot_date'] + pd.Timedelta(days=1)).sum()),
    'periodi di bilancio successivi al filing': int((data['report_date'] > data['accepted_at']).sum()),
    'unit audit non ok': int((data['unit_audit_status'] != 'ok').sum()),
    'valute miste': int(data['mixed_currency'].fillna(False).astype(bool).sum()),
}
for horizon in ('1m', '3m', '1y'):
    label_date = pd.to_datetime(data[f'label_date_{horizon}'], utc=True).dt.tz_localize(None)
    mature = data[f'target_{horizon}'].notna()
    checks[f'label {horizon} non futura'] = int(((label_date <= data['snapshot_date']) & mature).sum())
    checks[f'target raw incoerente {horizon}'] = int(
        ((data[f'target_{horizon}'] - data[f'target_raw_{horizon}']).abs() > 1e-12).fillna(False).sum()
    )
print(pd.Series(checks, name='violazioni').to_string())

In [ ]:
quality = data.groupby('filing_frequency').agg(
    snapshot=('ticker', 'size'),
    copertura_feature_mediana=('feature_coverage', 'median'),
    copertura_sorgente_mediana=('source_metric_coverage', 'median'),
    copertura_sorgente_minima=('source_metric_coverage', 'min'),
)
label_maturity = pd.Series({
    horizon: data[f'target_{horizon}'].notna().mean()
    for horizon in ('1m', '3m', '1y')
}, name='quota_label_mature')
print(quality.round(4).to_string())
print('\n', label_maturity.round(4).to_string())

### Gate del candidato

In [ ]:
import sys
sys.path.insert(0, str(backend))
from ml.model_governance import audit_candidate_artifact

audit = audit_candidate_artifact(artifact)
rows = []
for horizon, result in audit['horizons'].items():
    performance = artifact['models'][horizon]['performance']
    rows.append({
        'orizzonte': horizon,
        'decisione': result['decision'],
        'status': artifact['models'][horizon].get('validationStatus'),
        'MAE': performance.get('mae'),
        'baseline': performance.get('baselineZeroMae'),
        'IC cross-sectional': performance.get('crossSectionalRankIcMean'),
        'copertura 80%': performance.get('interval80Coverage'),
        'motivi': ' | '.join(result['reasons']),
    })
gate_table = pd.DataFrame(rows)
print('Decisione complessiva:', audit['decision'])
print(gate_table.round(4).to_string(index=False))

## Takeaways

1. La pipeline point-in-time supera i controlli strutturali principali: nessun duplicato alla grana dichiarata e nessun caso di look-ahead rilevato.
2. La copertura delle feature è buona in mediana, ma il percentile inferiore resta fragile e deve continuare a generare warning per singolo titolo.
3. Il candidato trimestrale non è pronto per una promozione completa: 1 mese e 3 mesi hanno blocker statistici. Il solo orizzonte annuale supera i gate.
4. Il blocker di qualità dati ancora aperto è l'universo manuale corrente: senza security master storico non si può escludere survivorship/delisting bias.
5. L'artifact live non viene sovrascritto; la prossima promozione richiede un candidato che superi tutti i gate congelati senza riutilizzare l'holdout per il tuning.